In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np

In [2]:
model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
    nn.ReLU(),
    nn.Linear(10, 2)
)

opt = torch.optim.Adam(model.parameters(), lr = 1e-3)

In [3]:
env = gym.make('CartPole-v1')

In [4]:
def passing_loop():
    states = [torch.tensor(env.reset()[0])]
    actions = []
    rewards = []

    while True:
        model_action = torch.softmax(model(states[-1]), dim=-1)
        actions.append(torch.multinomial(model_action, num_samples=1))

        new_state, rew, term, trun, info = env.step((actions[-1]).item())
        rewards.append(rew)
        states.append(torch.tensor(new_state))

        if term or trun:
            break

    return states[:-1], actions, rewards

In [5]:
def calculate_returns(rews, gamma):
    new_rets = [rews[-1]]
    rews = rews[::-1][1:]
    for a in rews:
        new_rets.append(new_rets[-1] * gamma + a)

    return new_rets[::-1]

In [6]:
def collect_batch(n_episodes, gamma=0.99):
    all_states, all_actions, all_returns = [], [], []

    for _ in range(n_episodes):
        states, actions, rewards = passing_loop()
        all_states += states
        all_actions += actions
        all_returns += calculate_returns(rewards, gamma)

    states = torch.stack(all_states)
    actions = torch.cat(all_actions)
    returns = torch.tensor(all_returns, dtype=torch.float32)

    advantages = (returns - returns.mean()) / (returns.std() + 1e-8)
    avg_len = len(states) / n_episodes
    return states, actions, advantages, avg_len

In [ ]:
def update(states, actions, advantages):
    logp = torch.log_softmax(model(states), dim=1)
    chosen_logp = logp[torch.arange(len(states)), actions]
    loss = -torch.mean(chosen_logp * advantages)

    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss.item()

In [11]:
history = []

for it in range(300):
    s, a, adv, avg_len = collect_batch(10)
    loss = update(s, a, adv)
    history.append(avg_len)

In [12]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML


def watch(policy, greedy=False):
    render_env = gym.make("CartPole-v1", render_mode="rgb_array")
    state, _ = render_env.reset()
    frames = []

    while True:
        with torch.no_grad():
            logits = policy(torch.tensor(state))
        if greedy:
            action = logits.argmax().item()
        else:
            action = torch.multinomial(torch.softmax(logits, dim=-1), 1).item()

        state, _, term, trun, _ = render_env.step(action)
        frames.append(render_env.render())
        if term or trun:
            break

    render_env.close()
    return frames


def show_frames(frames):
    fig = plt.figure(figsize=(5, 3))
    img = plt.imshow(frames[0])
    plt.axis("off")
    anim = animation.FuncAnimation(
        fig, lambda i: [img.set_data(frames[i]), img][1:], frames=len(frames), interval=30
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [13]:
frames = watch(model)
print(len(frames))
show_frames(frames)

frames = watch(model, greedy=True)
show_frames(frames)

500
